# Savant — ligand screening against a target

A two-stage cascade, scoped to one claim: **given a protein, rank a library of candidate
ligands.** Not "what is the absolute pKd", and not "which of two proteins binds tighter".

    stage 1   binder (pKd >= 6) vs non-binder        <- triage the library
    stage 2   pKd, trained on binders only           <- rank what survives

Encoders are fixed, not swept: **ESM-C 300M** (`Synthyra/ESMplusplus_small`) for the
protein and **ChemBERTa-77M-MLM** for the ligand. The sweeps are done — under a
leakage-controlled split every protein encoder landed within 0.04 of every other
(esmc-300m 0.462 to ankh-base 0.425, with 320-d esm2-8m statistically tied with 1536-d
ankh2-large), and no ligand encoder beat plain RDKit+ECFP4. Fixing them removes a
degree of freedom rather than losing one.

## Why this framing, in one table

The leakage-controlled 2x2 from the previous notebook, single-model control:

|  | familiar chemotype | novel chemotype |
|---|---|---|
| **seen protein** | 0.784 | **0.762** |
| **unseen protein** | 0.539 | 0.534 |

Novel chemistry costs ~0.02. A novel protein costs ~0.24. So the defensible product is
*new chemistry against a target you have data for* — which is exactly screening and lead
optimisation, and exactly what the v3 benchmark was built to measure: its own README says
it "measures ranking novel chemistry against an established target, not generalisation to
a new target. That is the realistic screening and lead-optimisation setting."

The benchmark also supports this on the merits: Savant's within-target r there is 0.407
against Boltz-2's 0.386, while its pooled r loses badly. Pooled r on that set is roughly
half a between-target test, and the between-target axis is close to unmeasurable there —
per-target corpus median pKd correlates with benchmark per-target mean at **r = -0.17**
across the 19 shared targets, because selection round-robins over pKd bins within each
target. A model is *penalised* for correctly learning a target's average level.

## What is different from the previous notebooks

- **Ranking metrics, not just correlation.** Per-target Spearman, ROC-AUC, enrichment
  factor and BEDROC — the numbers a screening campaign is actually judged on.
- **Real non-binders.** The 82,820 censor-cap rows (">10 uM") are the corpus's only
  genuine negatives and become stage-1 training data instead of being deleted.
- **Cascade coverage is reported.** v1's classifier rejected 288 of 935 benchmark pairs,
  every one a measured binder. Section 7 reports what stage 1 throws away, always.
- **The benchmark table names every tool.** Savant, Boltz-2, PRODIGY-LIG and PLAPT on the
  same 500 pairs and the same metrics.

Run from `ML_model/`. Embeddings and RDKit blocks are reused from `work_two_part/` if
present, so this does not re-embed anything.

In [1]:
import gc, json, time
from pathlib import Path

import numpy as np
import pandas as pd

HERE  = Path.cwd()
REPO  = HERE.parent
SRC   = REPO / "input" / "input.csv"
BENCH = HERE / "benchmark" / "eval" / "output" / "bench_scored.csv"
PREV  = HERE / "work_two_part"            # reuse its emb/ and static.npz
OUT   = HERE / "work_screening"
EMB   = PREV / "emb" if (PREV / "emb").exists() else OUT / "emb"
OUT.mkdir(parents=True, exist_ok=True); EMB.mkdir(parents=True, exist_ok=True)

SEED = 1337
MIN_PKD, MAX_PKD = 4.0, 12.0
MAX_LABEL_SD = 1.5
BINDER_PKD = 6.0
CENSOR_CAPS_NM = [10000.0, 100000.0, 1000000.0, 10000000.0]
USE_CAPS_AS_NEGATIVES = True

PROTEIN_ENC = dict(slug="esmc-300m", hf="Synthyra/ESMplusplus_small")
LIGAND_ENC  = dict(slug="chemberta-mlm77", hf="DeepChem/ChemBERTa-77M-MLM")

TEST_SCAFFOLD_FRAC = 0.20     # the screening condition: novel chemistry, known target
TEST_PROTEIN_FRAC  = 0.20     # reported as a secondary, harder cell
MIN_LIGANDS_PER_TARGET = 20   # a target needs a library before "screening" means anything
TARGET_RECALL = 0.95          # stage-1 threshold is tuned to this, not left at 0.5
N_THREADS = -1
ECFP_BITS = 1024
AA = "ACDEFGHIKLMNPQRSTVWY"

def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)
log(f"embeddings from {EMB}")

[21:46:42] embeddings from /scratch/sguirale/Savant/ML_model/work_two_part/emb


## 1. Label cleaning

Same as before, with one change of purpose: the censor caps are **stage-1 training data**
now, not waste. A `">10 uM"` record is not a pKd measurement, but it is a real statement
that the compound does not bind — which is precisely the negative class a screening
triage needs, and precisely what v1's classifier never had.

Factorisation runs over the cleaned measurements only, in that order, so the cached
`emb/*.npy` from `work_two_part/` stay aligned (21,140 proteins).

In [2]:
raw = pd.read_csv(SRC, usecols=["seq", "smiles", "neg_log10_affinity_M", "affinity_Kd_nM"])
log(f"{len(raw):,} rows in")
cap = np.zeros(len(raw), bool)
for c in CENSOR_CAPS_NM:
    cap |= np.isclose(raw.affinity_Kd_nM.to_numpy(np.float64), c, rtol=1e-6)
log(f"  censor-cap rows -> stage-1 negatives: {int(cap.sum()):,}")

caps_raw, meas = raw[cap], raw[~cap]
meas = meas[meas.neg_log10_affinity_M.between(MIN_PKD, MAX_PKD)].reset_index(drop=True)
meas["li"] = pd.factorize(meas.smiles)[0]
meas["pi"] = pd.factorize(meas.seq)[0]
ligands  = meas.drop_duplicates("li").sort_values("li").smiles.to_numpy()
proteins = meas.drop_duplicates("pi").sort_values("pi").seq.to_numpy()
log(f"  {len(proteins):,} proteins, {len(ligands):,} ligands  (must match the cached emb)")

lig_of = pd.Series(np.arange(len(ligands)), index=ligands)
pro_of = pd.Series(np.arange(len(proteins)), index=proteins)
caps_raw = caps_raw.assign(li=lig_of.reindex(caps_raw.smiles).to_numpy(),
                           pi=pro_of.reindex(caps_raw.seq).to_numpy()).dropna(subset=["li", "pi"])
caps_raw["li"] = caps_raw.li.astype(int); caps_raw["pi"] = caps_raw.pi.astype(int)

g = meas.groupby(["li", "pi"])["neg_log10_affinity_M"]
pairs = g.agg(pkd="median", n_meas="size", label_sd="std").reset_index()
pairs["label_sd"] = pairs.label_sd.fillna(0.0)
pairs = pairs[pairs.label_sd <= MAX_LABEL_SD].reset_index(drop=True)
pairs["w"] = np.sqrt(pairs.n_meas.to_numpy(np.float32))
caps = caps_raw.groupby(["li", "pi"]).size().reset_index(name="n_meas")
caps = caps[~caps.set_index(["li", "pi"]).index.isin(pairs.set_index(["li", "pi"]).index)]

log(f"{len(pairs):,} measured pairs; {len(caps):,} capped negatives")
log(f"  binders (pKd >= {BINDER_PKD}): {100*(pairs.pkd >= BINDER_PKD).mean():.1f}% of measured pairs")
log(f"  with caps included, positive rate becomes "
    f"{100*(pairs.pkd >= BINDER_PKD).sum()/(len(pairs)+len(caps)):.1f}%")
del raw, meas, caps_raw, g; gc.collect()

[21:46:56] 1,836,729 rows in


[21:46:56]   censor-cap rows -> stage-1 negatives: 82,820


[21:46:57]   21,140 proteins, 932,253 ligands  (must match the cached emb)


[21:46:58] 1,430,789 measured pairs; 43,917 capped negatives


[21:46:58]   binders (pKd >= 6.0): 67.2% of measured pairs


[21:46:58]   with caps included, positive rate becomes 65.2%


3

## 2. Feature blocks and embeddings

Reused from `work_two_part/` when present. Nothing here re-embeds unless the cache is
missing or the wrong shape.

In [3]:
from rdkit import Chem, RDLogger
from rdkit.Chem import (Crippen, Descriptors, Lipinski, rdFingerprintGenerator,
                        rdMolDescriptors)
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

def build_static():
    for cand in (PREV / "static.npz", OUT / "static.npz"):
        if cand.exists():
            z = np.load(cand, allow_pickle=True)
            if z["ecfp"].shape[0] == len(ligands):
                log(f"reusing {cand}")
                cols = json.loads((cand.parent / "rdkit_columns.json").read_text())
                return {k: z[k] for k in z.files}, cols
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=ECFP_BITS)
    d11, full, fps, scaf, ok = [], [], [], [], []
    for i, s in enumerate(ligands):
        m = Chem.MolFromSmiles(s); ok.append(m is not None)
        if m is None:
            d11.append([0.0]*11); full.append({}); fps.append(np.zeros(ECFP_BITS)); scaf.append("")
            continue
        mh = Chem.AddHs(m)
        d11.append([Descriptors.MolWt(mh), Crippen.MolLogP(mh), Lipinski.NumHDonors(mh),
                    Lipinski.NumHAcceptors(mh), Lipinski.NumRotatableBonds(mh),
                    rdMolDescriptors.CalcTPSA(mh), rdMolDescriptors.CalcNumRings(mh),
                    rdMolDescriptors.CalcNumAromaticRings(mh), mh.GetNumAtoms(),
                    mh.GetNumHeavyAtoms(), Chem.GetFormalCharge(mh)])
        full.append(Descriptors.CalcMolDescriptors(m))
        fps.append(gen.GetCountFingerprintAsNumPy(m))
        sc = MurckoScaffold.MurckoScaffoldSmiles(mol=m); scaf.append(sc if sc else Chem.MolToSmiles(m))
        if (i+1) % 100_000 == 0: log(f"  ligands {i+1:,}/{len(ligands):,}")
    fd = pd.DataFrame(full).astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    fd = fd.loc[:, fd.max() != fd.min()]
    aac = np.array([[s.count(a)/len(s) for a in AA] + [len(s)] for s in proteins], np.float32)
    d = dict(d11=np.asarray(d11, np.float32), dfull=fd.to_numpy(np.float32),
             ecfp=np.asarray(fps, np.float32), aac=aac,
             scaf=np.asarray(scaf), lig_ok=np.asarray(ok))
    np.savez_compressed(OUT / "static.npz", **d)
    (OUT / "rdkit_columns.json").write_text(json.dumps(list(fd.columns)))
    return d, list(fd.columns)

S, RD_COLS = build_static()
pairs = pairs[S["lig_ok"][pairs.li.to_numpy()]].reset_index(drop=True)
caps  = caps[S["lig_ok"][caps.li.to_numpy()]].reset_index(drop=True)
log(f"{len(pairs):,} pairs, {len(caps):,} caps after ligand parse filter")

[21:47:08] reusing /scratch/sguirale/Savant/ML_model/work_two_part/static.npz


[21:47:27] 1,430,784 pairs, 43,917 caps after ligand parse filter


In [4]:
def embed_texts(texts, hf, batch_size, max_length):
    import torch
    from transformers import AutoModel, AutoTokenizer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"  {hf} on {dev}, {len(texts):,} items")
    mdl = AutoModel.from_pretrained(hf, trust_remote_code=True).to(dev).eval()
    try:
        tok = AutoTokenizer.from_pretrained(hf, trust_remote_code=True)
    except Exception:
        tok = getattr(mdl, "tokenizer", None)
        if tok is None: raise
    order = np.argsort([len(t) for t in texts]); out = None
    with torch.inference_mode():
        for st in range(0, len(order), batch_size):
            sel = order[st:st+batch_size]
            b = tok([texts[i] for i in sel], padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt")
            b = {k: v.to(dev) for k, v in b.items()}
            h = mdl(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            pooled = ((h*m).sum(1)/m.sum(1).clamp_min(1)).float().cpu().numpy()
            if out is None: out = np.zeros((len(texts), pooled.shape[1]), np.float32)
            out[sel] = pooled
    del mdl
    if dev == "cuda": torch.cuda.empty_cache()
    gc.collect(); return out

def cached(enc, texts, batch_size, max_length):
    p = EMB / f"{enc['slug']}.npy"
    if p.exists():
        a = np.load(p, mmap_mode="r")
        if a.shape[0] == len(texts):
            log(f"  reuse {p.name} {a.shape}"); return np.asarray(a, np.float32)
        log(f"  {p.name} has {a.shape[0]} rows, expected {len(texts)} - rebuilding")
    arr = embed_texts(texts, enc["hf"], batch_size, max_length)
    np.save(p, arr); return arr

P_EMB = cached(PROTEIN_ENC, list(proteins), 8, 1024)
L_EMB = cached(LIGAND_ENC, list(ligands), 512, 256)
log(f"protein {P_EMB.shape}  ligand {L_EMB.shape}")

[21:47:27]   reuse esmc-300m.npy (21140, 960)


[21:47:27]   reuse chemberta-mlm77.npy (932253, 384)


[21:47:27] protein (21140, 960)  ligand (932253, 384)


## 3. Split — novel chemistry against known targets

The primary split holds out **scaffolds**, keeping targets, because that is the screening
condition and the one the benchmark measures. A protein-held-out cell is carried alongside
so the harder claim is never accidentally made: numbers from it are reported, not hidden.

    screening (primary)   novel scaffold, target seen in training
    new-target (harder)   novel scaffold AND novel protein

In [5]:
rng = np.random.default_rng(SEED)
pi_all = pairs.pi.to_numpy(); li_all = pairs.li.to_numpy()
y_all  = pairs.pkd.to_numpy(np.float32); w_all = pairs.w.to_numpy(np.float32)
scaf_u = pd.factorize(S["scaf"])[0]; sp_all = scaf_u[li_all]

us = np.unique(sp_all)
test_s = set(rng.choice(us, size=int(round(TEST_SCAFFOLD_FRAC*len(us))), replace=False).tolist())
per_p = pairs.groupby("pi").size()
bins = np.floor(np.log2(per_p.clip(lower=1))).astype(int)
test_p = []
for b in np.unique(bins):
    ps = per_p.index[bins == b].to_numpy()
    test_p.extend(rng.choice(ps, size=max(1, int(round(TEST_PROTEIN_FRAC*len(ps)))),
                             replace=False).tolist())
test_p = set(test_p)

ns = np.array([s in test_s for s in sp_all]); npr = np.array([p in test_p for p in pi_all])
tr_idx = np.where(~ns & ~npr)[0]
CELLS = {"screening (novel chem, known target)": np.where(ns & ~npr)[0],
         "new target (novel chem + novel protein)": np.where(ns & npr)[0]}
log(f"train {len(tr_idx):,} pairs / {len(np.unique(pi_all[tr_idx])):,} proteins")
for k, v in CELLS.items():
    log(f"  {k:42s} {len(v):>9,} pairs / {len(np.unique(pi_all[v])):>6,} targets")
assert not (set(sp_all[CELLS['screening (novel chem, known target)']]) & set(sp_all[tr_idx]))
assert not (set(pi_all[CELLS['new target (novel chem + novel protein)']]) & set(pi_all[tr_idx]))
log("no scaffold leak into either cell; no protein leak into the new-target cell")

## capped negatives split on the same axes
cap_sp = scaf_u[caps.li.to_numpy()]; cap_pi = caps.pi.to_numpy()
cap_ns = np.array([s in test_s for s in cap_sp]); cap_np = np.array([p in test_p for p in cap_pi])
cap_tr = np.where(~cap_ns & ~cap_np)[0]
CAP_CELLS = {"screening (novel chem, known target)": np.where(cap_ns & ~cap_np)[0],
             "new target (novel chem + novel protein)": np.where(cap_ns & cap_np)[0]}
log(f"capped negatives: train {len(cap_tr):,}, "
    f"screening {len(CAP_CELLS['screening (novel chem, known target)']):,}")

[21:47:28] train 899,650 pairs / 14,339 proteins


[21:47:28]   screening (novel chem, known target)         249,629 pairs /  8,017 targets


[21:47:28]   new target (novel chem + novel protein)       59,649 pairs /  1,990 targets


[21:47:29] no scaffold leak into either cell; no protein leak into the new-target cell


[21:47:29] capped negatives: train 27,759, screening 7,959


## 4. Screening metrics

Correlation is not how a screen is judged. These are:

- **per-target Spearman** — can you order this target's library?
- **per-target ROC-AUC** at pKd >= 6 — can you separate its binders from its non-binders?
- **EF@x%** — how many more hits in the top x% than picking at random. EF of 5 at 10%
  means five times the hit rate you would get by chance.
- **BEDROC (alpha=20)** — like ROC-AUC but weighted to the top of the list, which is the
  only part a screen ever looks at. ~80% of the weight sits in the top 8%.

All are computed **per target and then averaged**, never pooled, because pooling across
targets re-introduces the between-target axis this notebook is explicitly not claiming.

In [6]:
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score

def ef_at(y, s, frac):
    N, n = len(y), int(y.sum())
    if n == 0 or n == N: return np.nan
    k = max(1, int(round(frac*N)))
    return (y[np.argsort(-s)[:k]].sum()/k) / (n/N)

def bedroc(y, s, alpha=20.0):
    '''Truchon & Bayly (2007). 1.0 = all actives first, ~0 = all last.'''
    N, n = len(y), int(y.sum())
    if n == 0 or n == N: return np.nan
    ranks = np.where(y[np.argsort(-s)] == 1)[0] + 1
    Ra = n / N
    rie = np.sum(np.exp(-alpha*ranks/N)) / (Ra*(1-np.exp(-alpha))/(np.exp(alpha/N)-1))
    return (rie * Ra*np.sinh(alpha/2)/(np.cosh(alpha/2)-np.cosh(alpha/2-alpha*Ra))
            + 1.0/(1-np.exp(alpha*(1-Ra))))

def per_target(truth, score, groups, min_lig=MIN_LIGANDS_PER_TARGET, cut=BINDER_PKD):
    '''One row per target, then averaged. Targets too small to screen are dropped.

    Guards, both of which bit on the first run:
      * non-finite scores are dropped - sklearn's roc_auc_score rejects them outright,
        and a cascade that sends rejected compounds to -inf produces them by design.
      * a constant truth or score vector has no defined correlation; scipy warns and
        returns nan. Return nan deliberately instead of emitting a warning per target.
    '''
    d = pd.DataFrame({"y": truth, "s": score, "g": groups})
    d = d[np.isfinite(d.s.to_numpy()) & np.isfinite(d.y.to_numpy())]
    rows = []
    for gid, sub in d.groupby("g"):
        if len(sub) < min_lig: continue
        yv, sv = sub.y.to_numpy(), sub.s.to_numpy()
        lab = (yv >= cut).astype(int)
        const = (np.ptp(yv) == 0) or (np.ptp(sv) == 0)
        r = {"target": gid, "n": len(sub), "n_binders": int(lab.sum()),
             "spearman": np.nan if const else spearmanr(yv, sv).correlation}
        if 0 < lab.sum() < len(lab) and np.ptp(sv) > 0:
            r.update({"roc_auc": roc_auc_score(lab, sv), "pr_auc": average_precision_score(lab, sv),
                      "ef_10": ef_at(lab, sv, 0.10), "ef_20": ef_at(lab, sv, 0.20),
                      "bedroc": bedroc(lab, sv)})
        rows.append(r)
    return pd.DataFrame(rows)

def summarise(pt, tag):
    if pt.empty: return {"tag": tag, "n_targets": 0}
    out = {"tag": tag, "n_targets": len(pt), "median_lib_size": float(pt.n.median()),
           "spearman_mean": pt.spearman.mean(), "spearman_median": pt.spearman.median(),
           "frac_targets_positive": float((pt.spearman > 0).mean())}
    for c in ("roc_auc", "pr_auc", "ef_10", "ef_20", "bedroc"):
        if c in pt: out[c + "_mean"] = pt[c].mean()
    return out

## sanity: a perfect ranker and a random one, on a synthetic target
_y = np.array([1]*10 + [0]*90); _perf = np.r_[np.ones(10), np.zeros(90)]
_rng = np.random.default_rng(0); _rand = _rng.random(100)
print(f"sanity  perfect: EF@10 {ef_at(_y,_perf,0.10):.2f}  BEDROC {bedroc(_y,_perf):.3f}")
print(f"        random : EF@10 {ef_at(_y,_rand,0.10):.2f}  BEDROC {bedroc(_y,_rand):.3f}")

sanity  perfect: EF@10 10.00  BEDROC 1.000
        random : EF@10 0.00  BEDROC 0.040


## 5. Stage 1 — binder vs non-binder

Positives are measured pairs with pKd >= 6. Negatives are measured pairs below 6 **plus**
the capped `">10 uM"` rows, which are the only genuine non-binders in the corpus.

The threshold is tuned on a validation fold to `TARGET_RECALL`, not left at 0.5. That is
the specific defect that made v1's gate reject 288 measured binders: it trained on 94%
positives and then cut at 0.5.

In [7]:
import xgboost as xgb

def X_pairs(idx):
    return np.hstack([L_EMB[li_all[idx]], S["d11"][li_all[idx]], S["dfull"][li_all[idx]],
                      S["ecfp"][li_all[idx]], P_EMB[pi_all[idx]], S["aac"][pi_all[idx]]]
                     ).astype(np.float32)

def X_caps(idx):
    cl, cp = caps.li.to_numpy()[idx], caps.pi.to_numpy()[idx]
    return np.hstack([L_EMB[cl], S["d11"][cl], S["dfull"][cl], S["ecfp"][cl],
                      P_EMB[cp], S["aac"][cp]]).astype(np.float32)

vp = set(rng.choice(np.unique(pi_all[tr_idx]),
                    size=max(1, int(0.15*len(np.unique(pi_all[tr_idx])))), replace=False).tolist())
isv = np.array([p in vp for p in pi_all[tr_idx]])
fit_idx, val_idx = tr_idx[~isv], tr_idx[isv]
cap_isv = np.array([p in vp for p in caps.pi.to_numpy()[cap_tr]])
cap_fit, cap_val = cap_tr[~cap_isv], cap_tr[cap_isv]
log(f"fit {len(fit_idx):,} pairs + {len(cap_fit):,} caps | val {len(val_idx):,} + {len(cap_val):,}")

def clf_xy(idx, cidx):
    Xs = [X_pairs(idx)]; ys = [(y_all[idx] >= BINDER_PKD).astype(np.int8)]; ws = [w_all[idx]]
    if USE_CAPS_AS_NEGATIVES and len(cidx):
        Xs.append(X_caps(cidx)); ys.append(np.zeros(len(cidx), np.int8))
        ws.append(np.full(len(cidx), float(np.median(w_all[idx])), np.float32))
    return np.vstack(Xs), np.concatenate(ys), np.concatenate(ws)

Xf, yf, wf = clf_xy(fit_idx, cap_fit)
Xv, yv_, wv = clf_xy(val_idx, cap_val)
log(f"stage-1 train {Xf.shape}, positive rate {yf.mean():.3f}")
PARAMS = dict(max_depth=8, eta=0.05, subsample=0.8, colsample_bytree=0.5,
              min_child_weight=10, reg_lambda=2.0, tree_method="hist",
              nthread=N_THREADS, seed=SEED)
dtr = xgb.QuantileDMatrix(Xf, label=yf, weight=wf)
dva = xgb.QuantileDMatrix(Xv, label=yv_, ref=dtr)
del Xf; gc.collect()
clf = xgb.train({**PARAMS, "objective": "binary:logistic", "eval_metric": "aucpr"},
                dtr, num_boost_round=2000, evals=[(dva, "val")],
                early_stopping_rounds=50, verbose_eval=200)
pv = clf.inplace_predict(Xv)
THRESH = float(np.quantile(pv[yv_ == 1], 1 - TARGET_RECALL))
log(f"stage-1 val ROC {roc_auc_score(yv_, pv):.4f}  PR {average_precision_score(yv_, pv):.4f}")
log(f"threshold for {TARGET_RECALL:.0%} binder recall: {THRESH:.4f} "
    f"(keeps {100*(pv >= THRESH).mean():.1f}% of validation rows)")
del Xv, dtr, dva; gc.collect()

[21:47:31] fit 786,405 pairs + 23,823 caps | val 113,245 + 3,936


[21:47:45] stage-1 train (810228, 2615), positive rate 0.661


[0]	val-aucpr:0.71114


[125]	val-aucpr:0.77547


[21:54:23] stage-1 val ROC 0.7235  PR 0.7755


[21:54:23] threshold for 95% binder recall: 0.3783 (keeps 85.5% of validation rows)


66

## 6. Stage 2 — pKd on binders only

Trained on measured pairs with pKd >= 6, as specified.

**Read its correlations with the range restriction in mind.** Cutting at pKd 6 drops the
label spread from 1.399 to 0.990, so Pearson r on the binder subset is mechanically lower
than on the full range for an identical model — that is the correlation being computed over
a narrower window, not the model getting worse. This is exactly why section 4's ranking
metrics, not r, are the ones to quote for a screening claim.

In [8]:
bind_fit = fit_idx[y_all[fit_idx] >= BINDER_PKD]
bind_val = val_idx[y_all[val_idx] >= BINDER_PKD]
log(f"stage-2 train {len(bind_fit):,} binders, val {len(bind_val):,}")
log(f"  label sd on binders {y_all[bind_fit].std():.3f} vs {y_all[fit_idx].std():.3f} on all")

Xb = X_pairs(bind_fit)
dtr = xgb.QuantileDMatrix(Xb, label=y_all[bind_fit], weight=w_all[bind_fit]); del Xb; gc.collect()
Xbv = X_pairs(bind_val)
dva = xgb.QuantileDMatrix(Xbv, label=y_all[bind_val], ref=dtr); del Xbv; gc.collect()
reg = xgb.train({**PARAMS, "objective": "reg:squarederror", "eval_metric": "rmse"},
                dtr, num_boost_round=4000, evals=[(dva, "val")],
                early_stopping_rounds=50, verbose_eval=200)
del dtr, dva; gc.collect()
log(f"stage-2 best_iteration {getattr(reg, 'best_iteration', -1)}")

[21:54:23] stage-2 train 535,706 binders, val 73,367


[21:54:23]   label sd on binders 0.995 vs 1.399 on all


[0]	val-rmse:0.96545


[200]	val-rmse:0.91178


[214]	val-rmse:0.91140


[22:01:00] stage-2 best_iteration 164


## 7. Cascade evaluation

The screening score is stage 1's probability; the affinity estimate is stage 2's pKd on
what stage 1 keeps. Both are reported, plus **coverage** — how much of the library the gate
throws away and how many true binders go with it. v1 lost 288 measured binders that way
and the number never appeared next to its accuracy.

In [9]:
def evaluate(idx, cidx, tag):
    X = X_pairs(idx)
    p_bind = clf.inplace_predict(X)
    pk = reg.inplace_predict(X)
    del X; gc.collect()
    y, g = y_all[idx], pi_all[idx]
    lab = (y >= BINDER_PKD).astype(int)

    kept = p_bind >= THRESH
    rows = []
    ## screening: rank the whole library by P(binder)
    rows.append({**summarise(per_target(y, p_bind, g), f"{tag} | rank by P(binder)")})
    ## rank by predicted pKd instead
    rows.append({**summarise(per_target(y, pk, g), f"{tag} | rank by predicted pKd")})
    ## Cascade: gate, then rank survivors by pKd with rejects sent to the bottom.
    ## The floor must be FINITE - -inf is the natural way to write "last" but
    ## roc_auc_score refuses non-finite input, which killed this cell on the first run.
    floor = float(np.min(pk)) - 1.0 if len(pk) else 0.0
    casc = np.where(kept, pk, floor)
    rows.append({**summarise(per_target(y, casc, g), f"{tag} | cascade (gate + pKd)")})

    cov = {"tag": f"{tag} | COVERAGE", "n_targets": np.nan,
           "kept_frac": float(kept.mean()),
           "binders_kept": float(kept[lab == 1].mean()),
           "binders_lost": int((lab == 1).sum() - kept[lab == 1].sum()),
           "nonbinders_rejected": float(1 - kept[lab == 0].mean()) if (lab == 0).any() else np.nan}
    if USE_CAPS_AS_NEGATIVES and len(cidx):
        Xc = X_caps(cidx); pc = clf.inplace_predict(Xc); del Xc; gc.collect()
        cov["capped_negatives_rejected"] = float((pc < THRESH).mean())
    return rows, cov, per_target(y, p_bind, g)

all_rows, cov_rows, PT = [], [], {}
for cell, idx in CELLS.items():
    r, cov, pt = evaluate(idx, CAP_CELLS[cell], cell)
    all_rows += r; cov_rows.append(cov); PT[cell] = pt
    log(f"{cell}: {len(pt)} targets with >= {MIN_LIGANDS_PER_TARGET} ligands")

RES = pd.DataFrame(all_rows).set_index("tag")
COV = pd.DataFrame(cov_rows).set_index("tag")
RES.to_csv(OUT / "screening_internal.csv"); COV.to_csv(OUT / "cascade_coverage.csv")
display(RES.round(4)); display(COV.round(4))
for k, v in PT.items():
    v.to_csv(OUT / f"per_target_{k.split()[0]}.csv", index=False)

[22:01:18] screening (novel chem, known target): 1441 targets with >= 20 ligands


[22:01:22] new target (novel chem + novel protein): 361 targets with >= 20 ligands


,n_targets,median_lib_size,spearman_mean,spearman_median,frac_targets_positive,roc_auc_mean,pr_auc_mean,ef_10_mean,ef_20_mean,bedroc_mean
tag,,,,,,,,,,
"screening (novel chem, known target) | rank by P(binder)",1441,68.0,0.3014,0.3292,0.8466,0.7059,0.7434,1.4506,1.3658,0.7626
"screening (novel chem, known target) | rank by predicted pKd",1441,68.0,0.2862,0.3049,0.8126,0.6460,0.7083,1.2754,1.2307,0.7234
"screening (novel chem, known target) | cascade (gate + pKd)",1441,68.0,0.3181,0.3381,0.8203,0.6646,0.7228,1.3631,1.2853,0.7484
new target (novel chem + novel protein) | rank by P(binder),361,67.0,0.2166,0.2271,0.7867,0.6608,0.7133,1.4291,1.3193,0.7097
new target (novel chem + novel protein) | rank by predicted pKd,361,67.0,0.2178,0.2150,0.7645,0.6371,0.7029,1.4488,1.2963,0.7071
new target (novel chem + novel protein) | cascade (gate + pKd),361,67.0,0.2183,0.2218,0.7673,0.6365,0.7063,1.4789,1.2983,0.7099


,n_targets,kept_frac,binders_kept,binders_lost,nonbinders_rejected,capped_negatives_rejected
tag,,,,,,
"screening (novel chem, known target) | COVERAGE",NaN,0.8240,0.9492,8384,0.4206,0.3779
new target (novel chem + novel protein) | COVERAGE,NaN,0.8549,0.9457,2109,0.3144,0.2110


## 8. Benchmark — every tool, the same metrics

The 500-pair v3 benchmark, with Savant scored here and Boltz-2, PRODIGY-LIG and PLAPT read
from the columns `06_merge_scores.py` already wrote. Same targets, same ligands, same
metric code for all four, so the comparison is like-for-like.

Two caveats that belong next to the table:

- **It is a small screen.** 20 targets, ~25 ligands each, and **88.6% of pairs are binders
  at pKd 6**. A real screening deck is >99% inactive. Enrichment and BEDROC on a set this
  dense are optimistic and noisy; treat per-target Spearman as the primary number here and
  the enrichment columns as indicative.
- **Its targets are seen in training** — by design, since it measures novel chemistry
  against established targets. That is the claim this notebook makes, so the two match;
  it is not a zero-shot-on-target result and must not be described as one.

In [10]:
if not BENCH.exists():
    log(f"no benchmark at {BENCH}")
else:
    bm = pd.read_csv(BENCH)
    TRUTH = "neg_log10_affinity_M"
    ## This notebook's model against the three external tools, and nothing else. Earlier
    ## Savant generations are deliberately absent - the comparison being made is
    ## "the current model vs the field", and carrying four superseded variants turns the
    ## table into an internal changelog. Their numbers remain in bench_summary.md.
    TOOLS = {"Savant (this notebook)": None,          # filled in below
             "Boltz-2": "boltz_affinity_pic50",
             "PRODIGY-LIG": "prodigy_pkd",
             "PLAPT": "plapt_pk"}

    ## score this notebook's cascade on the benchmark
    keep = [Chem.MolFromSmiles(s) is not None for s in bm.ligand_smiles]
    bm = bm[keep].reset_index(drop=True)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=ECFP_BITS)
    d11b, fullb, fpb = [], [], []
    for s in bm.ligand_smiles:
        m = Chem.MolFromSmiles(s); mh = Chem.AddHs(m)
        d11b.append([Descriptors.MolWt(mh), Crippen.MolLogP(mh), Lipinski.NumHDonors(mh),
                     Lipinski.NumHAcceptors(mh), Lipinski.NumRotatableBonds(mh),
                     rdMolDescriptors.CalcTPSA(mh), rdMolDescriptors.CalcNumRings(mh),
                     rdMolDescriptors.CalcNumAromaticRings(mh), mh.GetNumAtoms(),
                     mh.GetNumHeavyAtoms(), Chem.GetFormalCharge(mh)])
        fullb.append(Descriptors.CalcMolDescriptors(m))
        fpb.append(gen.GetCountFingerprintAsNumPy(m))
    fdb = (pd.DataFrame(fullb).astype(np.float32).replace([np.inf, -np.inf], np.nan)
             .fillna(0.0).reindex(columns=RD_COLS, fill_value=0.0).to_numpy(np.float32))
    Lb = embed_texts(list(bm.ligand_smiles), LIGAND_ENC["hf"], 256, 256)
    Pb = embed_texts(list(bm.target_sequence), PROTEIN_ENC["hf"], 8, 1024)
    aab = np.array([[s.count(a)/len(s) + 0.0 for a in AA] + [len(s)]
                    for s in bm.target_sequence], np.float32)
    Xb = np.hstack([Lb, np.asarray(d11b, np.float32), fdb,
                    np.asarray(fpb, np.float32), Pb, aab]).astype(np.float32)
    bm["savant_screen_p"] = clf.inplace_predict(Xb)
    bm["savant_screen_pk"] = reg.inplace_predict(Xb)
    TOOLS["Savant (this notebook)"] = "savant_screen_p"

    y = bm[TRUTH].to_numpy(np.float32); g = bm.uniprot_id.to_numpy()
    lab = (y >= BINDER_PKD).astype(int)
    log(f"benchmark: {len(bm)} pairs, {bm.uniprot_id.nunique()} targets, "
        f"{100*lab.mean():.1f}% binders at pKd {BINDER_PKD}")

    rows, per_tool_pt = [], {}
    for name, col in TOOLS.items():
        if col is None or col not in bm or bm[col].isna().all():
            log(f"  {name}: column missing/empty, skipped"); continue
        s = bm[col].to_numpy(np.float64)
        ok = np.isfinite(s)
        pt = per_target(y[ok], s[ok], g[ok], min_lig=8)
        per_tool_pt[name] = pt
        row = summarise(pt, name)
        row["n_pairs"] = int(ok.sum())
        row["pooled_pearson"] = float(np.corrcoef(y[ok], s[ok])[0, 1])
        rows.append(row)
    BR = pd.DataFrame(rows).set_index("tag")
    cols = ["n_pairs", "n_targets", "spearman_mean", "spearman_median",
            "frac_targets_positive", "roc_auc_mean", "ef_20_mean", "bedroc_mean",
            "pooled_pearson"]
    BR = BR[[c for c in cols if c in BR]]
    BR.to_csv(OUT / "benchmark_screening.csv")
    display(BR.round(4))
    for n, pt in per_tool_pt.items():
        pt.to_csv(OUT / f"benchmark_per_target_{n.split()[0].lower()}.csv", index=False)
    print("\nspearman_mean is the headline: mean per-target rank correlation.")
    print("pooled_pearson is shown only for continuity with bench_summary.md - it mixes in")
    print("the between-target axis this notebook does not claim.")

[22:01:38]   DeepChem/ChemBERTa-77M-MLM on cuda, 500 items


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[22:01:40]   Synthyra/ESMplusplus_small on cuda, 500 items


Loading weights:   0%|          | 0/308 [00:00<?, ?it/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


[22:01:59] benchmark: 500 pairs, 20 targets, 88.6% binders at pKd 6.0


,n_pairs,n_targets,spearman_mean,spearman_median,frac_targets_positive,roc_auc_mean,ef_20_mean,bedroc_mean,pooled_pearson
tag,,,,,,,,,
Savant (this notebook),500,20,0.2566,0.3584,0.75,0.6945,1.0181,0.8190,0.1725
Boltz-2,496,20,0.2281,0.2418,0.75,0.6480,1.0716,0.8713,0.4120
PRODIGY-LIG,496,20,0.2328,0.1960,0.75,0.5858,1.0986,0.8707,0.3331
PLAPT,500,20,0.0672,0.1109,0.60,0.5892,0.9714,0.8315,0.0214



spearman_mean is the headline: mean per-target rank correlation.
pooled_pearson is shown only for continuity with bench_summary.md - it mixes in
the between-target axis this notebook does not claim.


## 9. Save

In [11]:
import joblib
bundle = {
    "variant": "screening_cascade",
    "protein_encoder": PROTEIN_ENC["hf"], "ligand_encoder": LIGAND_ENC["hf"],
    "block_order": ["ligand_emb", "d11", "rdkit_named", "ecfp", "protein_emb", "aa"],
    "rdkit_columns": RD_COLS, "ecfp_bits": ECFP_BITS,
    "classifier_json": clf.save_raw(raw_format="json"),
    "regressor_json": reg.save_raw(raw_format="json"),
    "binder_pkd": BINDER_PKD, "threshold": THRESH, "target_recall": TARGET_RECALL,
    "caps_as_negatives": USE_CAPS_AS_NEGATIVES,
    "split": {"mode": "scaffold-disjoint (screening)", "seed": SEED,
              "test_scaffold_frac": TEST_SCAFFOLD_FRAC, "test_protein_frac": TEST_PROTEIN_FRAC},
    "n_train_pairs": int(len(fit_idx)), "n_train_caps": int(len(cap_fit)),
    "internal": RES.reset_index().to_dict(orient="records"),
    "coverage": COV.reset_index().to_dict(orient="records"),
    "xgboost_version": xgb.__version__,
}
p = OUT / "savant_screening_esmc300_chemberta77.pkl"
joblib.dump(bundle, p, compress=3)
print(f"{p.name}  {p.stat().st_size/1e6:.0f} MB")
print(f"tables in {OUT}")

savant_screening_esmc300_chemberta77.pkl  3 MB
tables in /scratch/sguirale/Savant/ML_model/work_screening
